# 05. Scenario summary tables

This notebook computes scenario summary metrics: mean \(R(t)\), reduction in mean \(R(t)\), days with \(R(t)>1\), reduction in days with \(R(t)>1\), and excess transmission area. It supports manuscript Table 2 and Supplementary Tables S1-S4.


In [ ]:
# Repository path setup
# This cell makes the notebook runnable from either the repository root or the notebooks/ directory.
from pathlib import Path
import os


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "notebooks").exists():
            return candidate
    if current.name == "notebooks":
        return current.parent
    return current


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

for directory in [
    "data/metro",
    "data/NHIS/2016~2017",
    "data/mobility_factor/2016~2017",
    "data/Rt/2016~2017",
    "data/Rt/2017~2018",
    "data/Rt/2018~2019",
    "data/Rt/2022~2023",
    "figures/mobility_factor",
    "figures/2016~2017",
    "figures/HeatMap",
    "figures/validation",
    "results/validation",
    "results/tables",
]:
    Path(directory).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


# ============================================================
# 1. 파일 경로 직접 지정
# ============================================================

FILE_PATHS = [
    r"data/Rt/2016~2017/Seoul_Rt(2016~2017).csv",
    r"data/Rt/2017~2018/Seoul_Rt(2017~2018).csv",
    r"data/Rt/2018~2019/Seoul_Rt(2018~2019).csv",
    r"data/Rt/2022~2023/Seoul_Rt(2022~2023).csv",
]


# ============================================================
# 2. 기본 설정
# ============================================================

DATE_COL = "date"

# 기준 시나리오 R_t^(0)
BASELINE_COL = "Seoul_Rt"

# 이동량 감소 시나리오 R_t^(c)
SCENARIO_COLS = [
    "Seoul_Rt_1",
    "Seoul_Rt_2",
    "Seoul_Rt_3",
    "Seoul_Rt_4",
    "Seoul_Rt_5",
]

# 결과 저장 경로
OUTPUT_DIR = Path("results/tables/rt_metric_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_CSV_PATH = OUTPUT_DIR / "combined_rt_data.csv"
METRICS_CSV_PATH = OUTPUT_DIR / "rt_metrics_result.csv"
EXCEL_PATH = OUTPUT_DIR / "rt_metrics_result.xlsx"


# ============================================================
# 3. 파일 불러오기
# ============================================================

def read_csv_safely(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


required_cols = [DATE_COL, BASELINE_COL] + SCENARIO_COLS

dataframes = []

for path in FILE_PATHS:
    path = Path(path)
    
    df = read_csv_safely(path)
    
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{path.name} 파일에 필요한 컬럼이 없습니다: {missing_cols}")
    
    df = df[required_cols].copy()
    
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    
    for col in [BASELINE_COL] + SCENARIO_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    dataframes.append(df)


# ============================================================
# 4. 모든 기간 파일 하나로 통합
# ============================================================

combined_df = pd.concat(dataframes, ignore_index=True)

combined_df = combined_df.sort_values(DATE_COL).reset_index(drop=True)

print("통합 데이터 크기:", combined_df.shape)
display(combined_df.head())


# ============================================================
# 5. 기준 시나리오 적용 및 시나리오별 지표 계산
# ============================================================

def get_scenario_number(rt_col):
    if rt_col == BASELINE_COL:
        return 0
    return int(rt_col.replace(f"{BASELINE_COL}_", ""))


def calculate_metrics_from_combined_data(df):
    result_rows = []
    
    baseline_rt = df[BASELINE_COL]
    
    for rt_col in [BASELINE_COL] + SCENARIO_COLS:
        scenario_rt = df[rt_col]
        
        valid_mask = baseline_rt.notna() & scenario_rt.notna()
        
        R0 = baseline_rt[valid_mask]
        Rc = scenario_rt[valid_mask]
        
        n = len(Rc)
        
        # 9.1 평균 Rt
        mean_Rt = Rc.mean()
        
        # 9.2 평균 Rt 감소량
        mean_Rt_decrease = (R0 - Rc).mean()
        
        # 9.3 평균 상대 감소율
        mean_relative_decrease_percent = (
            ((R0 - Rc) / R0.replace(0, np.nan)) * 100
        ).mean()
        
        # 9.4 Rt > 1 일수
        D_c = int((Rc > 1).sum())
        
        # 기준 시나리오의 Rt > 1 일수
        D_0 = int((R0 > 1).sum())
        
        # 9.5 Rt > 1 일수 감소율
        if D_0 == 0:
            days_Rt_gt_1_reduction_percent = np.nan
        else:
            days_Rt_gt_1_reduction_percent = 100 * (D_0 - D_c) / D_0
        
        # 9.6 전파 초과 면적
        excess_transmission_area = np.maximum(Rc - 1, 0).sum()
        
        result_rows.append(
            {
                "scenario_c": get_scenario_number(rt_col),
                "rt_column": rt_col,
                "mean_Rt": mean_Rt,
                "mean_Rt_decrease": mean_Rt_decrease,
                "mean_relative_decrease_percent": mean_relative_decrease_percent,
                "days_Rt_gt_1": D_c,
                "days_Rt_gt_1_reduction_percent": days_Rt_gt_1_reduction_percent,
                "excess_transmission_area": excess_transmission_area,
            }
        )
    
    return pd.DataFrame(result_rows)


metrics_result = calculate_metrics_from_combined_data(combined_df)




# ============================================================
# 7. 계산 결과 보여주기
# ============================================================

print("저장 완료")
print("통합 데이터 CSV:", COMBINED_CSV_PATH)

display(metrics_result.round(6))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


FILE_PATHS = [
    r"data/Rt/2016~2017/Busan_Rt(2016~2017).csv",
    r"data/Rt/2017~2018/Busan_Rt(2017~2018).csv",
    r"data/Rt/2018~2019/Busan_Rt(2018~2019).csv",
    r"data/Rt/2022~2023/Busan_Rt(2022~2023).csv",
]



DATE_COL = "date"

# 기준 시나리오 R_t^(0)
BASELINE_COL = "Busan_Rt"

# 이동량 감소 시나리오 R_t^(c)
SCENARIO_COLS = [
    "Busan_Rt_1",
    "Busan_Rt_2",
    "Busan_Rt_3",
    "Busan_Rt_4",
    "Busan_Rt_5",
]

# 결과 저장 경로
OUTPUT_DIR = Path("results/tables/rt_metric_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_CSV_PATH = OUTPUT_DIR / "combined_rt_data.csv"
METRICS_CSV_PATH = OUTPUT_DIR / "rt_metrics_result.csv"
EXCEL_PATH = OUTPUT_DIR / "rt_metrics_result.xlsx"



def read_csv_safely(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


required_cols = [DATE_COL, BASELINE_COL] + SCENARIO_COLS

dataframes = []

for path in FILE_PATHS:
    path = Path(path)
    
    df = read_csv_safely(path)
    
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{path.name} 파일에 필요한 컬럼이 없습니다: {missing_cols}")
    
    df = df[required_cols].copy()
    
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    
    for col in [BASELINE_COL] + SCENARIO_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    dataframes.append(df)



combined_df = pd.concat(dataframes, ignore_index=True)

combined_df = combined_df.sort_values(DATE_COL).reset_index(drop=True)

print("통합 데이터 크기:", combined_df.shape)
display(combined_df.head())



def get_scenario_number(rt_col):
    if rt_col == BASELINE_COL:
        return 0
    return int(rt_col.replace(f"{BASELINE_COL}_", ""))


def calculate_metrics_from_combined_data(df):
    result_rows = []
    
    baseline_rt = df[BASELINE_COL]
    
    for rt_col in [BASELINE_COL] + SCENARIO_COLS:
        scenario_rt = df[rt_col]
        
        valid_mask = baseline_rt.notna() & scenario_rt.notna()
        
        R0 = baseline_rt[valid_mask]
        Rc = scenario_rt[valid_mask]
        
        n = len(Rc)
        
        # 9.1 평균 Rt
        mean_Rt = Rc.mean()
        
        # 9.2 평균 Rt 감소량
        mean_Rt_decrease = (R0 - Rc).mean()
        
        # 9.3 평균 상대 감소율
        mean_relative_decrease_percent = (
            ((R0 - Rc) / R0.replace(0, np.nan)) * 100
        ).mean()
        
        # 9.4 Rt > 1 일수
        D_c = int((Rc > 1).sum())
        
        # 기준 시나리오의 Rt > 1 일수
        D_0 = int((R0 > 1).sum())
        
        # 9.5 Rt > 1 일수 감소율
        if D_0 == 0:
            days_Rt_gt_1_reduction_percent = np.nan
        else:
            days_Rt_gt_1_reduction_percent = 100 * (D_0 - D_c) / D_0
        
        # 9.6 전파 초과 면적
        excess_transmission_area = np.maximum(Rc - 1, 0).sum()
        
        result_rows.append(
            {
                "scenario_c": get_scenario_number(rt_col),
                "rt_column": rt_col,
                "mean_Rt": mean_Rt,
                "mean_Rt_decrease": mean_Rt_decrease,
                "mean_relative_decrease_percent": mean_relative_decrease_percent,
                "days_Rt_gt_1": D_c,
                "days_Rt_gt_1_reduction_percent": days_Rt_gt_1_reduction_percent,
                "excess_transmission_area": excess_transmission_area,
            }
        )
    
    return pd.DataFrame(result_rows)


metrics_result = calculate_metrics_from_combined_data(combined_df)



display(metrics_result.round(6))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


FILE_PATHS = [
    r"data/Rt/2016~2017/Daegu_Rt(2016~2017).csv",
    r"data/Rt/2017~2018/Daegu_Rt(2017~2018).csv",
    r"data/Rt/2018~2019/Daegu_Rt(2018~2019).csv",
    r"data/Rt/2022~2023/Daegu_Rt(2022~2023).csv",
]



DATE_COL = "date"

# 기준 시나리오 R_t^(0)
BASELINE_COL = "Daegu_Rt"

# 이동량 감소 시나리오 R_t^(c)
SCENARIO_COLS = [
    "Daegu_Rt_1",
    "Daegu_Rt_2",
    "Daegu_Rt_3",
    "Daegu_Rt_4",
    "Daegu_Rt_5",
]

# 결과 저장 경로
OUTPUT_DIR = Path("results/tables/rt_metric_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_CSV_PATH = OUTPUT_DIR / "combined_rt_data.csv"
METRICS_CSV_PATH = OUTPUT_DIR / "rt_metrics_result.csv"
EXCEL_PATH = OUTPUT_DIR / "rt_metrics_result.xlsx"



def read_csv_safely(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


required_cols = [DATE_COL, BASELINE_COL] + SCENARIO_COLS

dataframes = []

for path in FILE_PATHS:
    path = Path(path)
    
    df = read_csv_safely(path)
    
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{path.name} 파일에 필요한 컬럼이 없습니다: {missing_cols}")
    
    df = df[required_cols].copy()
    
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    
    for col in [BASELINE_COL] + SCENARIO_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    dataframes.append(df)



combined_df = pd.concat(dataframes, ignore_index=True)

combined_df = combined_df.sort_values(DATE_COL).reset_index(drop=True)

print("통합 데이터 크기:", combined_df.shape)
display(combined_df.head())



def get_scenario_number(rt_col):
    if rt_col == BASELINE_COL:
        return 0
    return int(rt_col.replace(f"{BASELINE_COL}_", ""))


def calculate_metrics_from_combined_data(df):
    result_rows = []
    
    baseline_rt = df[BASELINE_COL]
    
    for rt_col in [BASELINE_COL] + SCENARIO_COLS:
        scenario_rt = df[rt_col]
        
        valid_mask = baseline_rt.notna() & scenario_rt.notna()
        
        R0 = baseline_rt[valid_mask]
        Rc = scenario_rt[valid_mask]
        
        n = len(Rc)
        
        # 9.1 평균 Rt
        mean_Rt = Rc.mean()
        
        # 9.2 평균 Rt 감소량
        mean_Rt_decrease = (R0 - Rc).mean()
        
        # 9.3 평균 상대 감소율
        mean_relative_decrease_percent = (
            ((R0 - Rc) / R0.replace(0, np.nan)) * 100
        ).mean()
        
        # 9.4 Rt > 1 일수
        D_c = int((Rc > 1).sum())
        
        # 기준 시나리오의 Rt > 1 일수
        D_0 = int((R0 > 1).sum())
        
        # 9.5 Rt > 1 일수 감소율
        if D_0 == 0:
            days_Rt_gt_1_reduction_percent = np.nan
        else:
            days_Rt_gt_1_reduction_percent = 100 * (D_0 - D_c) / D_0
        
        # 9.6 전파 초과 면적
        excess_transmission_area = np.maximum(Rc - 1, 0).sum()
        
        result_rows.append(
            {
                "scenario_c": get_scenario_number(rt_col),
                "rt_column": rt_col,
                "mean_Rt": mean_Rt,
                "mean_Rt_decrease": mean_Rt_decrease,
                "mean_relative_decrease_percent": mean_relative_decrease_percent,
                "days_Rt_gt_1": D_c,
                "days_Rt_gt_1_reduction_percent": days_Rt_gt_1_reduction_percent,
                "excess_transmission_area": excess_transmission_area,
            }
        )
    
    return pd.DataFrame(result_rows)


metrics_result = calculate_metrics_from_combined_data(combined_df)



display(metrics_result.round(6))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


FILE_PATHS = [
    r"data/Rt/2016~2017/Daejeon_Rt(2016~2017).csv",
    r"data/Rt/2017~2018/Daejeon_Rt(2017~2018).csv",
    r"data/Rt/2018~2019/Daejeon_Rt(2018~2019).csv",
    r"data/Rt/2022~2023/Daejeon_Rt(2022~2023).csv",
]



DATE_COL = "date"

# 기준 시나리오 R_t^(0)
BASELINE_COL = "Daejeon_Rt"

# 이동량 감소 시나리오 R_t^(c)
SCENARIO_COLS = [
    "Daejeon_Rt_1",
    "Daejeon_Rt_2",
    "Daejeon_Rt_3",
    "Daejeon_Rt_4",
    "Daejeon_Rt_5",
]

# 결과 저장 경로
OUTPUT_DIR = Path("results/tables/rt_metric_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_CSV_PATH = OUTPUT_DIR / "combined_rt_data.csv"
METRICS_CSV_PATH = OUTPUT_DIR / "rt_metrics_result.csv"
EXCEL_PATH = OUTPUT_DIR / "rt_metrics_result.xlsx"



def read_csv_safely(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


required_cols = [DATE_COL, BASELINE_COL] + SCENARIO_COLS

dataframes = []

for path in FILE_PATHS:
    path = Path(path)
    
    df = read_csv_safely(path)
    
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{path.name} 파일에 필요한 컬럼이 없습니다: {missing_cols}")
    
    df = df[required_cols].copy()
    
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    
    for col in [BASELINE_COL] + SCENARIO_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    dataframes.append(df)



combined_df = pd.concat(dataframes, ignore_index=True)

combined_df = combined_df.sort_values(DATE_COL).reset_index(drop=True)

print("통합 데이터 크기:", combined_df.shape)
display(combined_df.head())



def get_scenario_number(rt_col):
    if rt_col == BASELINE_COL:
        return 0
    return int(rt_col.replace(f"{BASELINE_COL}_", ""))


def calculate_metrics_from_combined_data(df):
    result_rows = []
    
    baseline_rt = df[BASELINE_COL]
    
    for rt_col in [BASELINE_COL] + SCENARIO_COLS:
        scenario_rt = df[rt_col]
        
        valid_mask = baseline_rt.notna() & scenario_rt.notna()
        
        R0 = baseline_rt[valid_mask]
        Rc = scenario_rt[valid_mask]
        
        n = len(Rc)
        
        # 9.1 평균 Rt
        mean_Rt = Rc.mean()
        
        # 9.2 평균 Rt 감소량
        mean_Rt_decrease = (R0 - Rc).mean()
        
        # 9.3 평균 상대 감소율
        mean_relative_decrease_percent = (
            ((R0 - Rc) / R0.replace(0, np.nan)) * 100
        ).mean()
        
        # 9.4 Rt > 1 일수
        D_c = int((Rc > 1).sum())
        
        # 기준 시나리오의 Rt > 1 일수
        D_0 = int((R0 > 1).sum())
        
        # 9.5 Rt > 1 일수 감소율
        if D_0 == 0:
            days_Rt_gt_1_reduction_percent = np.nan
        else:
            days_Rt_gt_1_reduction_percent = 100 * (D_0 - D_c) / D_0
        
        # 9.6 전파 초과 면적
        excess_transmission_area = np.maximum(Rc - 1, 0).sum()
        
        result_rows.append(
            {
                "scenario_c": get_scenario_number(rt_col),
                "rt_column": rt_col,
                "mean_Rt": mean_Rt,
                "mean_Rt_decrease": mean_Rt_decrease,
                "mean_relative_decrease_percent": mean_relative_decrease_percent,
                "days_Rt_gt_1": D_c,
                "days_Rt_gt_1_reduction_percent": days_Rt_gt_1_reduction_percent,
                "excess_transmission_area": excess_transmission_area,
            }
        )
    
    return pd.DataFrame(result_rows)


metrics_result = calculate_metrics_from_combined_data(combined_df)



display(metrics_result.round(6))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


FILE_PATHS = [
    r"data/Rt/2016~2017/Gwangju_Rt(2016~2017).csv",
    r"data/Rt/2017~2018/Gwangju_Rt(2017~2018).csv",
    r"data/Rt/2018~2019/Gwangju_Rt(2018~2019).csv",
    r"data/Rt/2022~2023/Gwangju_Rt(2022~2023).csv",
]



DATE_COL = "date"

# 기준 시나리오 R_t^(0)
BASELINE_COL = "Gwangju_Rt"

# 이동량 감소 시나리오 R_t^(c)
SCENARIO_COLS = [
    "Gwangju_Rt_1",
    "Gwangju_Rt_2",
    "Gwangju_Rt_3",
    "Gwangju_Rt_4",
    "Gwangju_Rt_5",
]

# 결과 저장 경로
OUTPUT_DIR = Path("results/tables/rt_metric_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_CSV_PATH = OUTPUT_DIR / "combined_rt_data.csv"
METRICS_CSV_PATH = OUTPUT_DIR / "rt_metrics_result.csv"
EXCEL_PATH = OUTPUT_DIR / "rt_metrics_result.xlsx"



def read_csv_safely(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


required_cols = [DATE_COL, BASELINE_COL] + SCENARIO_COLS

dataframes = []

for path in FILE_PATHS:
    path = Path(path)
    
    df = read_csv_safely(path)
    
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{path.name} 파일에 필요한 컬럼이 없습니다: {missing_cols}")
    
    df = df[required_cols].copy()
    
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    
    for col in [BASELINE_COL] + SCENARIO_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    dataframes.append(df)



combined_df = pd.concat(dataframes, ignore_index=True)

combined_df = combined_df.sort_values(DATE_COL).reset_index(drop=True)

print("통합 데이터 크기:", combined_df.shape)
display(combined_df.head())



def get_scenario_number(rt_col):
    if rt_col == BASELINE_COL:
        return 0
    return int(rt_col.replace(f"{BASELINE_COL}_", ""))


def calculate_metrics_from_combined_data(df):
    result_rows = []
    
    baseline_rt = df[BASELINE_COL]
    
    for rt_col in [BASELINE_COL] + SCENARIO_COLS:
        scenario_rt = df[rt_col]
        
        valid_mask = baseline_rt.notna() & scenario_rt.notna()
        
        R0 = baseline_rt[valid_mask]
        Rc = scenario_rt[valid_mask]
        
        n = len(Rc)
        
        # 9.1 평균 Rt
        mean_Rt = Rc.mean()
        
        # 9.2 평균 Rt 감소량
        mean_Rt_decrease = (R0 - Rc).mean()
        
        # 9.3 평균 상대 감소율
        mean_relative_decrease_percent = (
            ((R0 - Rc) / R0.replace(0, np.nan)) * 100
        ).mean()
        
        # 9.4 Rt > 1 일수
        D_c = int((Rc > 1).sum())
        
        # 기준 시나리오의 Rt > 1 일수
        D_0 = int((R0 > 1).sum())
        
        # 9.5 Rt > 1 일수 감소율
        if D_0 == 0:
            days_Rt_gt_1_reduction_percent = np.nan
        else:
            days_Rt_gt_1_reduction_percent = 100 * (D_0 - D_c) / D_0
        
        # 9.6 전파 초과 면적
        excess_transmission_area = np.maximum(Rc - 1, 0).sum()
        
        result_rows.append(
            {
                "scenario_c": get_scenario_number(rt_col),
                "rt_column": rt_col,
                "mean_Rt": mean_Rt,
                "mean_Rt_decrease": mean_Rt_decrease,
                "mean_relative_decrease_percent": mean_relative_decrease_percent,
                "days_Rt_gt_1": D_c,
                "days_Rt_gt_1_reduction_percent": days_Rt_gt_1_reduction_percent,
                "excess_transmission_area": excess_transmission_area,
            }
        )
    
    return pd.DataFrame(result_rows)


metrics_result = calculate_metrics_from_combined_data(combined_df)



display(metrics_result.round(6))